In [18]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
import operator

load_dotenv()

llm = ChatGroq(model="llama-3.3-70b-versatile")

In [13]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Feedback on the essay")
    score: int = Field(description = "Score out of 10", ge=0, le=10)

structured_model = llm.with_structured_output(EvaluationSchema)

In [14]:
essay = """ India is a country in South Asia. It is the seventh-largest country by land area, the second-most populous country, and the most populous democracy in the world. Bounded by the Indian Ocean on the south, the Arabian Sea on the southwest, and the Bay of Bengal on the southeast, it shares land borders with Pakistan to the west; China, Nepal, and Bhutan to the north; and Bangladesh and Myanmar to the east. In the Indian Ocean, India is in the vicinity of Sri Lanka and the Maldives; its Andaman and Nicobar Islands share a maritime border with Thailand and Indonesia.
India's diverse culture, rich history, and varied geography make it a unique and fascinating country. The country has a long and complex history, with ancient civilizations, empires, and colonial influences shaping its development. India is known for its contributions to art, science, mathematics, and philosophy. It is also famous for its festivals, cuisine, and traditional clothing.
The country has a federal parliamentary democratic system, with a President as the head of state and a  Prime Minister as the head of government. India is a member of various international organizations, including the United Nations, the World Trade Organization, and the Commonwealth of Nations. It has a mixed economy, with agriculture, manufacturing, and services sectors contributing to its GDP.
India faces various challenges, including poverty, corruption, and environmental issues. However, it has made significant progress in areas such as technology, education, and healthcare. The country continues to strive for economic growth, social development, and political stability, while preserving its cultural heritage and diversity."""

In [19]:
class EssayState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_score: Annotated[list[int], operator.add] # reducer function to merge individual scores into a single list
    avg_score: float

In [26]:
def eval_lang(state: EssayState) -> EssayState:
    prompt = f"Evaluate the language quality of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"
    res = structured_model.invoke(prompt)
    
    return {'language_feedback': res.feedback, 'individual_score': [res.score]}

def eval_analysis(state: EssayState) -> EssayState:
    prompt = f"Evaluate the analysis quality of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"
    res = structured_model.invoke(prompt)
    
    return {'analysis_feedback': res.feedback, 'individual_score': [res.score]}

def eval_clarity(state: EssayState) -> EssayState:
    prompt = f"Evaluate the clarity of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"
    res = structured_model.invoke(prompt)
    
    return {'clarity_feedback': res.feedback, 'individual_score': [res.score]}

def final_eval(state: EssayState) -> EssayState:
    prompt = f"Based on following feedback:\nLanguage - {state['language_feedback']}\nAnalysis - {state['analysis_feedback']}\nClarity - {state['clarity_feedback']}\n, Generate an overall feedback."
    overall_feedback = llm.invoke(prompt).content
    avg_score = sum(state['individual_score']) / len(state['individual_score'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [27]:
graph = StateGraph(EssayState)

graph.add_node('eval_lang',eval_lang)
graph.add_node('eval_analysis',eval_analysis)
graph.add_node('eval_clarity',eval_clarity)
graph.add_node('final_eval',final_eval)

graph.add_edge(START, 'eval_lang')
graph.add_edge(START, 'eval_analysis')
graph.add_edge(START, 'eval_clarity')
graph.add_edge('eval_lang', 'final_eval')
graph.add_edge('eval_analysis', 'final_eval')
graph.add_edge('eval_clarity', 'final_eval')
graph.add_edge('final_eval', END)

workflow = graph.compile()

In [29]:
final_state = workflow.invoke({'essay': essay})
print(final_state['overall_feedback'])
print(final_state['avg_score'])

**Overall Feedback**

This essay provides a solid foundation for exploring the complexities of India, covering its geography, culture, history, and current situation in a clear and concise manner. The writing is formal and objective, with a logical structure that is easy to follow. However, to take the essay to the next level, it is essential to delve deeper into the topics, providing more nuanced and detailed analysis, as well as supporting evidence to substantiate the points being made.

To improve, consider the following key areas:

1. **Enhance vocabulary and sentence structure**: Incorporate more varied and sophisticated vocabulary to add depth and complexity to the writing. Additionally, break up long sentences to improve clarity and flow.
2. **Provide specific examples and supporting details**: Include concrete examples and evidence to make the information more engaging, convincing, and memorable.
3. **Develop critical thinking and analysis**: Move beyond stating facts and inste